# Cross-Device Transformer (HuggingFace PatchTST) — CPU Temperature Forecast

This notebook replaces the CNN+GRU baseline (`cross_device_gru_cnn_forecast_20_tuned.ipynb`)
with a Transformer — HuggingFace's **PatchTST** (`PatchTSTForRegression`) — while keeping the
**exact same data pipeline**, so results are directly comparable:

- Same 10 per-device CSV files, same 6 feature columns.
- Same cleaning (winsorized noisy columns, dropped implausible sensor values).
- Same phase-grouping (`mode` → `phaseGroup`) and **10-minute contiguous time blocks**.
- Same **block-balanced train/val/test split**, stratified by `(runName, phaseGroup)` — this is
  what "combines" the different log files: each device's data is cut into contiguous blocks,
  blocks (not raw rows) are split across train/val/test, and only *then* are sequences built
  independently within each block and pooled into one big array across all devices. This keeps a
  device's own held-out time period from leaking into its own training data, and keeps the
  batches during `model.fit`-equivalent training randomly mixed across devices.
- Same target definition: a **single** dT value (`future_temp - current_temp`) at a 20s horizon
  (10 samples), from a 40s lookback window (20 samples) — a direct regression, not a multi-step
  sequence forecast.
- Same evaluation: reconstruct absolute future temperature (`current + predicted_dT`), report
  MAE / RMSE / R² against the actual future temperature, compare to a persistence baseline, and
  break results down by experiment phase.

**What's different:** the model. `PatchTSTForRegression` patches each of the 6 channels
independently and encodes them with a *shared* Transformer encoder (channel-independence — the
same idea that makes PatchTST state-of-the-art for long-horizon forecasting), then a regression
head maps the fused representation to a single dT value.


In [1]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import joblib
import random

from transformers import PatchTSTConfig, PatchTSTForRegression

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Configuration

Identical lookback/horizon/interval to the CNN+GRU baseline.

In [3]:
# Approximate sensor interval
SAMPLE_INTERVAL_SECONDS = 2

LOOKBACK_SECONDS = 26      # 40s lookback -> 20 samples
FORECAST_SECONDS = 10      # 20s horizon -> 10 samples ahead

LOOKBACK = LOOKBACK_SECONDS // SAMPLE_INTERVAL_SECONDS
HORIZON = FORECAST_SECONDS // SAMPLE_INTERVAL_SECONDS

print("Lookback samples:", LOOKBACK)
print("Forecast horizon samples:", HORIZON)

FEATURE_COLS = [
    "cpuUsage",
    "cpuPackagePower",
    "gpuCoreTemperature",
    "gpuHotspotTemperature",
    "cpuEfficiencyAverageClock",
    "targetLoad",
    "cpuTemperature"
]

BLOCK_MINUTES = 10
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

BATCH_SIZE = 32
EPOCHS = 100
LR = 8e-4
PATIENCE = 10
SAVE_PATH = "patchtst_regression_best_1sec.pt"

Lookback samples: 13
Forecast horizon samples: 5


## 2. Loading Datasets

Same set of per-device CSVs as the baseline. Point `FILES` at your actual data directory.

In [4]:
FILES = {
    "manan": "manan_merged_experiment.csv",
    "prabhsimrat": "prabh_merged_experiment.csv",
    "gursimar": "gursimar_merged_experiment.csv",
    "sushant": "sushant_merged_experiment.csv",
}

datasets = {}
for name, path in FILES.items():
    df = pd.read_csv(path)
    df["targetLoad"] = df["targetLoad"].fillna(0)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["source"] = name
    datasets[name] = df
    print(name, "-", df.shape)

manan - (27732, 29)
prabhsimrat - (28375, 29)
gursimar - (22022, 29)
sushant - (26424, 29)


## 3. Cleaning

Same rules as the baseline: drop implausible sensor readings, winsorize noisy columns (clip, don't drop, to keep the time index contiguous).

In [5]:
def clean_run(df):
    df = df.copy()
    df = df.dropna(subset=FEATURE_COLS)

    # Drop physically implausible sensor glitches
    df = df[(df["cpuTemperature"] > 0) & (df["cpuTemperature"] < 110)]
    df = df[(df["cpuUsage"] >= 0) & (df["cpuUsage"] <= 100)]

    # Winsorize (clip) remaining noisy features to their 1st-99th percentile
    for col in ["ramUsage", "networkConnections", "processCount", "cpuPackagePower"]:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    return df.sort_values("timestamp").reset_index(drop=True)


for name in datasets:
    datasets[name] = clean_run(datasets[name])
    print(name, len(datasets[name]))

manan 27732
prabhsimrat 28375
gursimar 22022
sushant 26424


## 4. Phase Grouping + Contiguous 10-Minute Blocks

This is the step that governs how log files get combined into batches later: each run is split into contiguous same-phase segments, then chopped into 10-minute blocks. Blocks — not raw rows — are the unit that gets assigned to train/val/test.

In [6]:
def get_phase_group(mode):
    mode = str(mode).upper()
    if "COOLING" in mode:
        return "COOLING"
    if mode in ["INITIAL_IDLE", "PRE_EXPERIMENT", "POST_EXPERIMENT"]:
        return "IDLE"
    if mode in ["RAMP", "CHAOS", "TRANSITION", "MIXED"]:
        return mode
    return "OTHER"


for name, df in datasets.items():
    df = df.copy()
    df["phaseGroup"] = df["mode"].apply(get_phase_group)
    datasets[name] = df


def create_time_blocks(df, run_name, block_minutes=10):
    df = df.sort_values("timestamp").reset_index(drop=True).copy()
    all_blocks = []
    block_counter = 0

    phase_change = df["phaseGroup"] != df["phaseGroup"].shift()
    df["phaseSegment"] = phase_change.cumsum()

    for segment_id, segment in df.groupby("phaseSegment"):
        segment = segment.sort_values("timestamp").copy()
        segment_start = segment["timestamp"].min()
        elapsed_minutes = (segment["timestamp"] - segment_start).dt.total_seconds() / 60
        segment["localBlock"] = (elapsed_minutes // block_minutes).astype(int)

        for local_block, block in segment.groupby("localBlock"):
            block = block.copy()
            block["blockId"] = f"{run_name}{block_counter}"
            block["runName"] = run_name
            all_blocks.append(block)
            block_counter += 1

    return all_blocks


all_blocks = []
for name, df in datasets.items():
    run_blocks = create_time_blocks(df, run_name=name, block_minutes=BLOCK_MINUTES)
    all_blocks.extend(run_blocks)
    print(name, "blocks:", len(run_blocks))

manan blocks: 57
prabhsimrat blocks: 59
gursimar blocks: 84
sushant blocks: 54


## 5. Drop Blocks Too Short to Build a Sequence

In [7]:
MIN_REQUIRED_SAMPLES = LOOKBACK + HORIZON + 5

valid_blocks = [b for b in all_blocks if len(b) >= MIN_REQUIRED_SAMPLES]
print("Valid blocks:", len(valid_blocks), "| Removed short blocks:", len(all_blocks) - len(valid_blocks))

block_summary = pd.DataFrame([
    {
        "blockId": b["blockId"].iloc[0],
        "runName": b["runName"].iloc[0],
        "phaseGroup": b["phaseGroup"].iloc[0],
        "samples": len(b),
    }
    for b in valid_blocks
])
print(block_summary["phaseGroup"].value_counts())

Valid blocks: 250 | Removed short blocks: 4
phaseGroup
OTHER         61
MIXED         60
RAMP          32
CHAOS         32
TRANSITION    32
COOLING       29
IDLE           4
Name: count, dtype: int64


## 6. Block-Balanced Train / Val / Test Split

Split *blocks* (not rows), grouped by `(runName, phaseGroup)`, so every device and every experiment phase is represented in all three splits — this is what prevents one device's or one phase's data from leaking across the split.

In [8]:
def split_blocks_balanced(block_summary, seed=42):
    train_ids, val_ids, test_ids = [], [], []
    grouped = block_summary.groupby(["runName", "phaseGroup"])
    rng = np.random.default_rng(seed)

    for (run_name, phase), group in grouped:
        ids = group["blockId"].tolist()
        rng.shuffle(ids)
        n = len(ids)

        if n == 1:
            train_ids.extend(ids)
            continue
        if n == 2:
            train_ids.append(ids[0])
            val_ids.append(ids[1])
            continue

        n_train = max(1, int(round(n * TRAIN_RATIO)))
        n_val = max(1, int(round(n * VAL_RATIO)))
        if n_train + n_val >= n:
            n_train = n - 2
            n_val = 1

        train_ids.extend(ids[:n_train])
        val_ids.extend(ids[n_train:n_train + n_val])
        test_ids.extend(ids[n_train + n_val:])

    return train_ids, val_ids, test_ids


train_block_ids, val_block_ids, test_block_ids = split_blocks_balanced(block_summary, seed=SEED)
print("Train blocks:", len(train_block_ids), "| Val blocks:", len(val_block_ids), "| Test blocks:", len(test_block_ids))

train_blocks, val_blocks, test_blocks = [], [], []
for block in valid_blocks:
    block_id = block["blockId"].iloc[0]
    if block_id in train_block_ids:
        train_blocks.append(block)
    elif block_id in val_block_ids:
        val_blocks.append(block)
    elif block_id in test_block_ids:
        test_blocks.append(block)

print("Train/Val/Test blocks:", len(train_blocks), len(val_blocks), len(test_blocks))

Train blocks: 179 | Val blocks: 33 | Test blocks: 38
Train/Val/Test blocks: 179 33 38


## 7. Fit Scaler on Training Blocks Only

In [9]:
scaler_training_data = pd.concat([b[FEATURE_COLS] for b in train_blocks], ignore_index=True)

scaler_X = StandardScaler()
scaler_X.fit(scaler_training_data)
joblib.dump(scaler_X, "cross_device_feature_scaler_transformer_1sec.pkl")
print("Scaler fitted only on training blocks.")

Scaler fitted only on training blocks.


## 8. Build Sequences Within Each Block

Same rules as the baseline: a sequence is only built if there's no gap greater than 5 seconds
anywhere across the lookback window *and* the horizon (sequences never cross a block boundary,
so they never cross a device or a large time gap either). Target is a single dT value:
`future_temp - current_temp`, `HORIZON` steps ahead.


In [10]:
def create_sequences_from_block(block, scaler, lookback, horizon, max_gap_seconds=5):
    block = block.sort_values("timestamp").reset_index(drop=True).copy()
    scaled_features = scaler.transform(block[FEATURE_COLS])
    temperatures = block["cpuTemperature"].to_numpy()
    timestamps = block["timestamp"].to_numpy()

    X, y = [], []
    current_temperatures, future_temperatures, prediction_timestamps = [], [], []

    for i in range(lookback, len(block) - horizon + 1):
        sequence_start = i - lookback
        current_index = i - 1
        future_index = current_index + horizon
        if future_index >= len(block):
            break

        relevant_times = block["timestamp"].iloc[sequence_start:future_index + 1]
        gaps = relevant_times.diff().dt.total_seconds().dropna()
        if (gaps > max_gap_seconds).any():
            continue

        X.append(scaled_features[sequence_start:i])

        current_temp = temperatures[current_index]
        future_temp = temperatures[future_index]
        y.append(future_temp - current_temp)

        current_temperatures.append(current_temp)
        future_temperatures.append(future_temp)
        prediction_timestamps.append(timestamps[future_index])

    return (
        np.asarray(X), np.asarray(y),
        np.asarray(current_temperatures), np.asarray(future_temperatures),
        np.asarray(prediction_timestamps),
    )


def create_dataset_from_blocks(blocks, scaler):
    all_X, all_y, all_current, all_future, all_timestamps = [], [], [], [], []
    all_phases, all_runs = [], []

    for block in blocks:
        X, y, current, future, timestamps = create_sequences_from_block(
            block=block, scaler=scaler, lookback=LOOKBACK, horizon=HORIZON,
        )
        if len(X) == 0:
            continue

        phase = block["phaseGroup"].iloc[0]
        run_name = block["runName"].iloc[0]

        all_X.append(X)
        all_y.append(y)
        all_current.append(current)
        all_future.append(future)
        all_timestamps.append(timestamps)
        all_phases.extend([phase] * len(X))
        all_runs.extend([run_name] * len(X))

    return (
        np.concatenate(all_X), np.concatenate(all_y),
        np.concatenate(all_current), np.concatenate(all_future),
        np.concatenate(all_timestamps), np.asarray(all_phases), np.asarray(all_runs),
    )

In [11]:
(X_train, y_train, train_current, train_actual, train_timestamps, train_phases, train_runs) = \
    create_dataset_from_blocks(train_blocks, scaler_X)

# Winsorize TRAINING target only (transient sensor glitches, not real 20s swings).
# Validation/test targets stay untouched so evaluation metrics remain honest.
y_train_lower = np.percentile(y_train, 1)
y_train_upper = np.percentile(y_train, 99)
print(f"Clipping training dT targets to [{y_train_lower:.3f}, {y_train_upper:.3f}] deg C (1st-99th pct)")
y_train = np.clip(y_train, y_train_lower, y_train_upper)

(X_val, y_val, val_current, val_actual, val_timestamps, val_phases, val_runs) = \
    create_dataset_from_blocks(val_blocks, scaler_X)

(X_test, y_test, test_current, test_actual, test_timestamps, test_phases, test_runs) = \
    create_dataset_from_blocks(test_blocks, scaler_X)

print("\nTRAIN X:", X_train.shape, "y:", y_train.shape)
print("VAL   X:", X_val.shape, "y:", y_val.shape)
print("TEST  X:", X_test.shape, "y:", y_test.shape)

Clipping training dT targets to [-15.500, 15.600] deg C (1st-99th pct)

TRAIN X: (70507, 13, 7) y: (70507,)
VAL   X: (14454, 13, 7) y: (14454,)
TEST  X: (15285, 13, 7) y: (15285,)


## 9. Model: HuggingFace `PatchTSTForRegression`

`PatchTSTConfig` sets up channel-independent patching (each of the 6 features patched and
encoded with shared Transformer weights, the core PatchTST idea) plus a regression head that
maps the fused per-channel representations to a single dT value (`num_targets=1`).

`scaling=None` because features are already standardized externally with `scaler_X` (fit on
train only) — letting PatchTST's own internal instance-normalization run on top would
double-normalize.


In [12]:
config = PatchTSTConfig(
    num_input_channels=len(FEATURE_COLS),
    context_length=LOOKBACK,
    patch_length=4,
    patch_stride=4,
    num_targets=1,
    d_model=64,
    num_attention_heads=4,
    num_hidden_layers=3,
    ffn_dim=128,
    dropout=0.2,
    head_dropout=0.2,
    channel_attention=True,   # let the 6 correlated channels attend to each other
    scaling=None,             # features already standardized externally
    loss="mse",
)

model = PatchTSTForRegression(config).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

Model parameters: 101,761


## 10. Dataset / DataLoader

In [13]:
class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32)).unsqueeze(-1)  # (N, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(RegressionDataset(X_train, y_train), batch_size=BATCH_SIZE,
                           shuffle=True, drop_last=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(RegressionDataset(X_val, y_val), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(RegressionDataset(X_test, y_test), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 11. Training

AdamW + ReduceLROnPlateau + early stopping + best-checkpoint saving, mirroring the baseline's callbacks. Mixed precision when a GPU is available.

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)

use_amp = DEVICE.type == "cuda"
amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n = 0.0, 0

    with torch.set_grad_enabled(is_train):
        for X, y in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad()
                with torch.amp.autocast("cuda", enabled=use_amp):
                    out = model(past_values=X, target_values=y)
                    loss = out.loss
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                with torch.amp.autocast("cuda", enabled=use_amp):
                    out = model(past_values=X, target_values=y)
                    loss = out.loss

            total_loss += loss.item() * X.size(0)
            n += X.size(0)

    return total_loss / n

In [15]:
best_val = float("inf")
patience_ctr = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    val_loss = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:03d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

    if val_loss < best_val:
        best_val = val_loss
        patience_ctr = 0
        torch.save({"model_state": model.state_dict(), "config": config.to_dict()}, SAVE_PATH)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 001 | train_loss=12.00714 | val_loss=13.15990
Epoch 002 | train_loss=11.11409 | val_loss=13.33894
Epoch 003 | train_loss=10.81124 | val_loss=13.13449
Epoch 004 | train_loss=10.72028 | val_loss=12.55955
Epoch 005 | train_loss=10.63810 | val_loss=12.87400
Epoch 006 | train_loss=10.53875 | val_loss=12.39731
Epoch 007 | train_loss=10.43408 | val_loss=12.38099
Epoch 008 | train_loss=10.46110 | val_loss=12.82956
Epoch 009 | train_loss=10.36413 | val_loss=12.38491
Epoch 010 | train_loss=10.33199 | val_loss=12.77822
Epoch 011 | train_loss=10.22367 | val_loss=12.32353
Epoch 012 | train_loss=10.23050 | val_loss=12.27200
Epoch 013 | train_loss=10.16585 | val_loss=12.30190
Epoch 014 | train_loss=10.10711 | val_loss=12.35438
Epoch 015 | train_loss=10.05957 | val_loss=12.27681
Epoch 016 | train_loss=9.99877 | val_loss=12.22655
Epoch 017 | train_loss=9.97457 | val_loss=12.02618
Epoch 018 | train_loss=9.93792 | val_loss=12.06576
Epoch 019 | train_loss=9.88396 | val_loss=12.01806
Epoch 020 | trai

## 12. Evaluation

Same methodology as the baseline: reconstruct absolute future temperature
(`current_temp + predicted_dT`) and score against the actual future temperature — not against
dT directly — with MAE, RMSE, and R², plus a persistence baseline and a per-phase breakdown.


In [16]:
ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

test_predicted_delta = []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = model(past_values=X)
        test_predicted_delta.append(out.regression_outputs.float().cpu().numpy())

test_predicted_delta = np.concatenate(test_predicted_delta).flatten()

# Reconstruct future temperature
test_transformer_predictions = test_current + test_predicted_delta

# Persistence baseline: assume no change
test_persistence = test_current.copy()

In [17]:
def calculate_metrics(actual, predicted, name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(f"\n===== {name} =====")
    print(f"MAE:  {mae:.3f} deg C")
    print(f"RMSE: {rmse:.3f} deg C")
    print(f"R2:   {r2:.4f}")

    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}


internal_baseline = calculate_metrics(test_actual, test_persistence, "Internal Test - Persistence")
internal_transformer = calculate_metrics(test_actual, test_transformer_predictions, "Internal Test - PatchTST")


===== Internal Test - Persistence =====
MAE:  2.179 deg C
RMSE: 4.577 deg C
R2:   0.9132

===== Internal Test - PatchTST =====
MAE:  1.725 deg C
RMSE: 3.654 deg C
R2:   0.9447


In [18]:
def evaluate_by_phase(actual, predicted, phases, model_name):
    results = []
    for phase in np.unique(phases):
        mask = phases == phase
        if mask.sum() < 10:
            continue
        mae = mean_absolute_error(actual[mask], predicted[mask])
        rmse = np.sqrt(mean_squared_error(actual[mask], predicted[mask]))
        results.append({
            "Model": model_name, "Phase": phase, "Samples": mask.sum(),
            "MAE": mae, "RMSE": rmse,
        })
    return pd.DataFrame(results)


transformer_phase_results = evaluate_by_phase(test_actual, test_transformer_predictions, test_phases, "PatchTST")
baseline_phase_results = evaluate_by_phase(test_actual, test_persistence, test_phases, "Persistence")

phase_comparison = pd.concat([baseline_phase_results, transformer_phase_results], ignore_index=True)
phase_comparison.sort_values(["Phase", "Model"])

,Model,Phase,Samples,MAE,RMSE
6,PatchTST,CHAOS,1978,2.017049,4.118999
0,Persistence,CHAOS,1978,2.402073,4.890448
7,PatchTST,COOLING,953,1.809839,2.920340
1,Persistence,COOLING,953,2.511962,4.705750
8,PatchTST,MIXED,5372,1.413606,3.268079
2,Persistence,MIXED,5372,1.728165,4.054832
9,PatchTST,OTHER,3066,2.137400,4.043828
3,Persistence,OTHER,3066,3.024070,5.264362
10,PatchTST,RAMP,1955,1.577661,3.299222
4,Persistence,RAMP,1955,1.781228,3.987354


## 13. (Optional) Compare directly against the CNN+GRU baseline

If you have `test_gru_predictions` and `test_actual` saved from the baseline notebook (same
`test_blocks`/`test_runs` split, since both notebooks use the same seed and split logic), you
can load them here and diff MAE/RMSE/R² side by side.


In [ ]:
# Example — uncomment and point at your saved baseline outputs:
# gru_predictions = np.load("gru_test_predictions.npy")
# gru_metrics = calculate_metrics(test_actual, gru_predictions, "Internal Test - CNN+GRU (baseline)")
#
# comparison = pd.DataFrame([internal_baseline, gru_metrics, internal_transformer])
# comparison

In [19]:
from google.colab import files

files.download('cross_device_feature_scaler_transformer_1sec.pkl')
files.download('patchtst_regression_best_1sec.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>